# PII Analysis on App Review Dataset

**Purpose:** Quantify Personally Identifiable Information (PII) in `AppReviewData.csv`,
design a masking strategy, and validate the implementation.

**Phase:** 3 — Construction

## Why This Matters

Review text is unstructured user-generated content. Users sometimes embed identifying
information (emails, phone numbers, addresses) in reviews — often when angry or seeking
direct response from app developers. Training models on this raw text means:

- PII gets baked into model weights and logs (privacy risk)
- Sharing the dataset becomes legally sensitive (GDPR, CCPA)
- Future audits cannot easily prove user data was handled properly

This notebook quantifies the problem and prototypes the masking solution.

In [3]:
import pandas as pd
import re
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

sns.set_style("whitegrid")
pd.set_option('display.max_colwidth', 300)

df = pd.read_csv(r"C:\Users\tb6353\Downloads\AppReview.csv")
df.head(3)

,appID,reviewerName,reviewText,reviewerRating,reviewDate,textAnalytics
0,3,Eric Hansen,"Love it! WELL worth the money for the full version. I came for the ad-blocking, and stayed for everything else. It really is as quick as lightning, super light on system resources, and easy to navigate. I do miss some of the gestures from Chrome (particularly the ""swipe down to view tabs"" gestur...",1.0,42923,NaN
1,3,Jacob N.,"There's an awful bug that doesn't allow you to use the space bar if you want to type a search term from the url box. Ruined the whole experience for me. There's free browsers like Dee Browser that are the same as this one, but without awful bugs. I can't believe this a paid app Full Review",0.4,42976,NaN
2,3,Higgins Family,Would be 5 stars except for the bugs.... For example in incognito tab you cannot type in an address without the cursor jumping out of the field. Had made it unusable Full Review,0.8,43010,NaN


## PII Categories We Will Scan For

Based on common patterns in user-generated content and the EDA findings:

| Category | Why It Matters | Detection Approach |
|---|---|---|
| Email addresses | Direct identifier; baseline preprocessor already removes these | Regex |
| Phone numbers | Direct identifier; not handled by baseline | Regex |
| URLs | May contain personal handles or invite links | Regex |
| Reviewer name | Already in a separate column; flag for handling | Column check |
| Money references | Quasi-identifier (refund requests + amounts) | Regex (informational only — not masked) |
| All-caps name patterns | Possible self-references like "JOHN HERE" | Heuristic |

Out of scope for this prototype: SSN, credit card, full home address. Patterns 
exist for these but our EDA found no obvious occurrences in app reviews.

In [4]:
# PII detection patterns
# Conservative — prefers false negatives over false positives
# (better to miss some than mask legitimate text incorrectly)

PII_PATTERNS = {
    'email': re.compile(
        r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b'
    ),
    
    # Matches: 555-123-4567, (555) 123-4567, 555.123.4567, +1-555-123-4567
    # Avoids matching version numbers like 1.2.3 by requiring 7+ digits total
    'phone': re.compile(
        r'(?:\+?\d{1,3}[\s.-]?)?\(?\d{3}\)?[\s.-]?\d{3}[\s.-]?\d{4}\b'
    ),
    
    # http(s)://... or www....
    'url': re.compile(
        r'(?:https?://|www\.)[^\s,;]+',
        re.IGNORECASE
    ),
    
    # @username patterns (Twitter/Instagram-style handles)
    'handle': re.compile(
        r'(?<![A-Za-z0-9])@[A-Za-z0-9_]{3,30}\b'
    ),
    
    # Money amounts (informational; we observe but don't mask these)
    'money': re.compile(
        r'\$\s?\d+(?:[.,]\d+)?|\b\d+\s?(?:dollars?|usd|euros?|gbp|inr|rs)\b',
        re.IGNORECASE
    ),
}

print("Patterns compiled. Categories:")
for name in PII_PATTERNS:
    print(f"  - {name}")

Patterns compiled. Categories:
  - email
  - phone
  - url
  - handle
  - money


In [6]:
def count_pii_in_text(text, patterns):
    """Return a dict of {category: count} for one review."""
    if pd.isna(text):
        return {k: 0 for k in patterns}
    counts = {}
    for name, pattern in patterns.items():
        counts[name] = len(pattern.findall(str(text)))
    return counts

# Apply to all reviews — this takes ~10 seconds
pii_counts = df['reviewText'].apply(lambda t: count_pii_in_text(t, PII_PATTERNS))
pii_df = pd.DataFrame(pii_counts.tolist())

# Combine back with original
df_with_pii = pd.concat([df.reset_index(drop=True), pii_df], axis=1)

# Summary stats
print("PII OCCURRENCE SUMMARY")
for cat in PII_PATTERNS:
    total_hits = pii_df[cat].sum()
    reviews_with = (pii_df[cat] > 0).sum()
    pct = reviews_with / len(df) * 100
    print(f"{cat:10s}  →  {total_hits:6,} total hits  |  "
          f"{reviews_with:5,} reviews ({pct:.2f}%)")

PII OCCURRENCE SUMMARY
email       →       3 total hits  |      3 reviews (0.00%)
phone       →      10 total hits  |      8 reviews (0.01%)
url         →       5 total hits  |      5 reviews (0.00%)
handle      →      72 total hits  |     66 reviews (0.06%)
money       →     187 total hits  |    164 reviews (0.15%)


In [7]:
# Show actual matches per category — verify our patterns are working correctly
print("=" * 70)
print("SAMPLE MATCHES — VERIFY PATTERNS ARE CATCHING REAL PII")
print("=" * 70)

for category, pattern in PII_PATTERNS.items():
    print(f"\n--- {category.upper()} ---")
    
    # Find reviews with at least one match
    matched_reviews = df[df['reviewText'].apply(
        lambda t: bool(pattern.search(str(t))) if pd.notna(t) else False
    )]
    
    if len(matched_reviews) == 0:
        print("  (no matches found)")
        continue
    
    # Show up to 5 examples
    for idx, row in matched_reviews.head(5).iterrows():
        text = str(row['reviewText'])[:200]
        matches = pattern.findall(str(row['reviewText']))
        print(f"  Match: {matches}")
        print(f"  Context: {text}")
        print()

SAMPLE MATCHES — VERIFY PATTERNS ARE CATCHING REAL PII

--- EMAIL ---
  Match: ['Tato.daf@gmail.xom']
  Context: Tato.daf@gmail.xom Interesting Full Review

  Match: ['gvancushka_magradze@yahoo.com']
  Context: HTC inspire Verafrit ver davayene verc fontebi da verc klaviatura.iqneb moaxerxot chemi damxmareba fontebis dayenebashi mainc da klaviaturas memgoni gavartmev tavs. Arc rootis arsi mesmis simartle rom

  Match: ['bhuvaneshsekar29101995@gmail.com']
  Context: Its a threat.... Its a virus it makes my new rooted device into loop reboot so dont think it helps you... if some one have any problem contact me bhuvaneshsekar29101995@gmail.com.... i have recovered 


--- PHONE ---
  Match: ['00000000000']
  Context: 000000000 00000000000 Full Review

  Match: ['0170412104109']
  Context: this app can prevert stamptime on my device 4.4.2 and 4.4.4 but cannot prevert stamptime when install my device 5.1. so please fix this bug. log: MFMT 20170412104109 Program.txt 500 unable to modify l

  